<a href="https://colab.research.google.com/github/mentlana/Analysis-of-symbols-and-standards-in-Muse/blob/main/%D0%AD%D1%82%D0%B0%D0%BF_1_%D0%A4%D0%BE%D1%80%D0%BC%D0%B8%D1%80%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D0%B5_%D0%BA%D0%BE%D1%80%D0%BF%D1%83%D1%81%D0%B0_%D0%B8_%D0%BF%D1%80%D0%B5%D0%B4%D0%BE%D0%B1%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%BA%D0%B0_%D0%AD%D1%82%D0%B0%D0%BF_1_%D0%A4%D0%BE%D1%80%D0%BC%D0%B8%D1%80%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D0%B5_%D0%BA%D0%BE%D1%80%D0%BF%D1%83%D1%81%D0%B0_%D0%B8_%D0%BF%D1%80%D0%B5%D0%B4%D0%BE%D0%B1%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%BA%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Этап 1. Формирование корпуса и предобработка

Импорт необходимых библиотек

In [1]:
import pandas as pd
from collections import Counter
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.corpus import stopwords
import string
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

Удаляем пустые строки, то есть строки, где нет текстового содержимого в колонке lyrics_raw

In [2]:
df = pd.read_csv("muse_corpus.csv")
df = df[df['lyrics_raw'].notna()]
df = df[df['lyrics_raw'].str.strip() != ""]
df = df.reset_index(drop=True)

In [3]:
print(f"Количество строк после удаления: {len(df)}")
total_words = df['lyrics_raw'].str.split().str.len().sum()
print(f"Всего слов в колонке lyrics_raw: {total_words}")

Количество строк после удаления: 106
Всего слов в колонке lyrics_raw: 18620


Очистка и предобработка текстов: удаление характерной для Genius.com разметки текста [Verse], [Chorus], лишних переносов, изолированных цифр, пунктуации и приведение текста к нижнему регистру.

In [4]:
def clean_lyrics(text):
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\b\d+\b', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = text.lower()
    return text

df['lyrics_clean'] = df['lyrics_raw'].apply(clean_lyrics)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 106 entries, 0 to 105
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         106 non-null    object
 1   album         106 non-null    object
 2   year          106 non-null    int64 
 3   lyrics_raw    106 non-null    object
 4   lyrics_clean  106 non-null    object
dtypes: int64(1), object(4)
memory usage: 4.3+ KB


In [5]:
print(df.shape)
df.head(5)

(106, 5)


,title,album,year,lyrics_raw,lyrics_clean
0,Sunburn,Showbiz,1999,[Verse 1]\nCome waste your millions here\nSecr...,\ncome waste your millions here\nsecretly she ...
1,Muscle Museum,Showbiz,1999,[Verse 1]\nShe had something to confess to\nBu...,\nshe had something to confess to\nbut you don...
2,Fillip,Showbiz,1999,"[Verse 1]\nIt's happening soon, it's happening...",\nits happening soon its happening soon\nits s...
3,Falling Down,Showbiz,1999,[Verse 1]\nI'm falling down\nAnd fifteen thous...,\nim falling down\nand fifteen thousand people...
4,Cave,Showbiz,1999,"[Verse 1]\nLeave me alone, it's nothing seriou...",\nleave me alone its nothing serious\nill do i...


Токенизация (+удаление стоп слов) и лемматизация текстов

In [10]:
extra_stopwords = {'woah', 'hey', 'ooh', 'yeah', 'ah', 'oh', 'la', 'na'}
stop_words = set(stopwords.words('english')).union(extra_stopwords)

lemmatizer = WordNetLemmatizer()
def tokenize_and_lemmatize(text):
    tokens = word_tokenize(text)
    tokens = [token for token in tokens if token not in stop_words]
    lemmas = [lemmatizer.lemmatize(token) for token in tokens]
    return lemmas

df['tokens'] = df['lyrics_clean'].apply(tokenize_and_lemmatize)
df['tokens_str'] = df['tokens'].apply(' '.join)
df['lemma_count'] = df['tokens'].apply(len)

In [8]:
df.to_csv('muse_corpus_full.csv', index=False, encoding='utf-8')
print("Сохранён: muse_corpus_full.csv")

with open('muse_corpus_for_antconc.txt', 'w', encoding='utf-8') as f:
    for lyrics in df['lyrics_clean']:
        f.write(lyrics + '\n\n')
print("Сохранён: muse_corpus_for_antconc.txt")

Сохранён: muse_corpus_full.csv
Сохранён: muse_corpus_for_antconc.txt


В результате на выходе два файла:
*   muse_corpus_full.csv – полная таблица с метаданными, очищенными текстами и токенами.
*   muse_corpus_for_antconc.txt – для анализа в корпусных менеджерах (AntConc).

In [12]:
print("\nСТАТИСТИКА КОРПУСА")
print(f"Всего песен: {len(df)}")
print(f"Средняя длина текста песни: {df['lemma_count'].mean():.1f}")
print(f"Всего слов: {df['lemma_count'].sum()}")
print(f"Общее количество уникальных лемм: {len(set([lemma for sublist in df['tokens'] for lemma in sublist]))}")


СТАТИСТИКА КОРПУСА
Всего песен: 106
Средняя длина текста песни: 82.5
Всего слов: 8750
Общее количество уникальных лемм: 1786
